# Spotter — USDA FoodData Central normalization

This notebook normalizes **Foundation Foods**, **SR Legacy**, and **FNDDS** into the shape needed by Spotter's `Food` model.

It deliberately separates:

1. **Structural normalization** — deterministic and safe.
2. **EDA / curation** — duplicates, long USDA names, categories, and catalog selection.

### Expected folder structure

Extract the three USDA CSV archives into:

```text
data/
└── raw/
    └── usda/
        ├── foundation/
        ├── sr_legacy/
        └── fndds/
```

The notebook searches recursively inside each folder, so the USDA archive may contain an extra nested directory.

### Outputs

```text
data/processed/usda_foods_stage.csv
data/processed/foods_for_db.csv
data/processed/usda_foods_rejected.csv
data/processed/usda_duplicate_name_candidates.csv
```

`foods_for_db.csv` intentionally omits `id`, `created_at`, and `updated_at`; Prisma/PostgreSQL should create those values.


In [ ]:
from pathlib import Path
import json
import re
import unicodedata

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

# Adjust this if the notebook lives somewhere else.
PROJECT_ROOT = Path.cwd()

RAW_ROOT = PROJECT_ROOT / "data" / "raw" / "usda"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCES = {
    "FOUNDATION": {
        "directory": RAW_ROOT / "foundation",
        "source": "USDA_FDC_FOUNDATION",
        "source_version": "2026-04-30",
    },
    "SR_LEGACY": {
        "directory": RAW_ROOT / "sr_legacy",
        "source": "USDA_FDC_SR_LEGACY",
        "source_version": "2018-04",
    },
    "FNDDS": {
        "directory": RAW_ROOT / "fndds",
        "source": "USDA_FDC_FNDDS",
        "source_version": "2024-10-31",
    },
}

# USDA FoodData Central nutrient IDs we need.
NUTRIENT_IDS = {
    "protein": 1003,
    "fat": 1004,
    "carbohydrate": 1005,
    "energy_legacy": 1008,
    "energy_atwater_general": 2047,
    "energy_atwater_specific": 2048,
}

TARGET_NUTRIENT_IDS = set(NUTRIENT_IDS.values())

# FNDDS food_nutrient.csv can be large, so read it in chunks.
NUTRIENT_CHUNK_SIZE = 500_000

# Do NOT automatically collapse same-name foods across USDA sources yet.
# First inspect the duplicate-candidate output.
DEDUPLICATE_EXACT_NAMES = False


## 1. Inspect what USDA actually gave us

USDA archives contain several relational CSV files. We use the core master food table, nutrient table, and source-specific category/identity tables.

This cell is intentionally first: if USDA changes a filename in a future release, you will see it before the pipeline fails.


In [ ]:
def list_csv_files(directory: Path):
    if not directory.exists():
        print(f"❌ Missing directory: {directory}")
        return []

    files = sorted(directory.rglob("*.csv"))
    print(f"\n{directory} — {len(files)} CSV files")
    for path in files:
        print("  ", path.relative_to(directory))
    return files


for source_name, config in SOURCES.items():
    print(f"\n===== {source_name} =====")
    list_csv_files(config["directory"])


## 2. File/column helpers

The helpers are a little defensive because USDA field naming has changed slightly between releases.

We fail loudly when a required column is missing instead of silently producing wrong nutrition.


In [ ]:
def find_csv(directory: Path, filename: str, required: bool = True):
    matches = [
        path for path in directory.rglob("*.csv")
        if path.name.lower() == filename.lower()
    ]

    if not matches:
        if required:
            raise FileNotFoundError(
                f"Could not find {filename!r} anywhere under {directory}"
            )
        return None

    if len(matches) > 1:
        raise RuntimeError(
            f"Found more than one {filename!r} under {directory}: {matches}"
        )

    return matches[0]


def read_csv_columns(path: Path):
    return list(pd.read_csv(path, nrows=0).columns)


def pick_column(columns, candidates, *, required=True, context=""):
    by_lower = {str(column).lower(): column for column in columns}

    for candidate in candidates:
        found = by_lower.get(candidate.lower())
        if found is not None:
            return found

    if required:
        raise KeyError(
            f"{context}: none of {candidates} found. Available columns: {columns}"
        )

    return None


def clean_text(value):
    if pd.isna(value):
        return None

    value = unicodedata.normalize("NFKC", str(value))
    value = re.sub(r"\s+", " ", value).strip()
    return value or None


def normalized_name_key(value):
    """Only for duplicate detection; never shown to the user."""
    value = clean_text(value)
    if not value:
        return None

    value = value.casefold()
    value = re.sub(r"[^\w\s]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def normalize_join_key(series):
    """Makes category code joins stable when one CSV reads codes as 100 and another as 100.0."""
    numeric = pd.to_numeric(series, errors="coerce")
    result = numeric.astype("Int64").astype("string")

    # If a value was not numeric, preserve the cleaned original.
    original = series.astype("string").str.strip()
    return result.where(numeric.notna(), original)


## 3. Load the four nutrition values

FoodData Central stores nutrient values in **long format**:

```text
fdc_id | nutrient_id | amount
```

We keep only six possible nutrient IDs and pivot them.

For energy, the priority is:

1. `2048` — Atwater specific factor
2. `2047` — Atwater general factor
3. `1008` — legacy Energy

SR Legacy/FNDDS normally use `1008`; newer Foundation records use the newer Atwater energy representations.


In [ ]:
def load_target_nutrients(directory: Path, wanted_fdc_ids=None):
    nutrient_path = find_csv(directory, "food_nutrient.csv")
    columns = read_csv_columns(nutrient_path)

    fdc_col = pick_column(columns, ["fdc_id"], context=str(nutrient_path))
    nutrient_col = pick_column(columns, ["nutrient_id"], context=str(nutrient_path))
    amount_col = pick_column(columns, ["amount"], context=str(nutrient_path))

    wanted_fdc_ids = set(wanted_fdc_ids) if wanted_fdc_ids is not None else None

    kept_chunks = []

    for chunk in pd.read_csv(
        nutrient_path,
        usecols=[fdc_col, nutrient_col, amount_col],
        chunksize=NUTRIENT_CHUNK_SIZE,
        low_memory=False,
    ):
        chunk = chunk.rename(columns={
            fdc_col: "fdc_id",
            nutrient_col: "nutrient_id",
            amount_col: "amount",
        })

        chunk["fdc_id"] = pd.to_numeric(chunk["fdc_id"], errors="coerce").astype("Int64")
        chunk["nutrient_id"] = pd.to_numeric(
            chunk["nutrient_id"], errors="coerce"
        ).astype("Int64")
        chunk["amount"] = pd.to_numeric(chunk["amount"], errors="coerce")

        chunk = chunk[chunk["nutrient_id"].isin(TARGET_NUTRIENT_IDS)]

        if wanted_fdc_ids is not None:
            chunk = chunk[chunk["fdc_id"].isin(wanted_fdc_ids)]

        if not chunk.empty:
            kept_chunks.append(chunk)

    if not kept_chunks:
        return pd.DataFrame(columns=[
            "fdc_id",
            "calories_per_100g",
            "protein_grams_per_100g",
            "carbohydrate_grams_per_100g",
            "fat_grams_per_100g",
            "_energy_nutrient_id",
        ])

    nutrients = pd.concat(kept_chunks, ignore_index=True)

    # A food should normally have one value per nutrient ID.
    # aggfunc="first" protects the pivot from failing while we separately report duplicates.
    duplicate_nutrients = nutrients.duplicated(
        subset=["fdc_id", "nutrient_id"], keep=False
    )
    if duplicate_nutrients.any():
        print(
            "⚠️ Repeated food/nutrient rows found:",
            duplicate_nutrients.sum(),
            "rows. Pivot will use the first value."
        )

    wide = nutrients.pivot_table(
        index="fdc_id",
        columns="nutrient_id",
        values="amount",
        aggfunc="first",
    ).reset_index()

    # Make sure every expected nutrient ID exists as a column.
    for nutrient_id in TARGET_NUTRIENT_IDS:
        if nutrient_id not in wide.columns:
            wide[nutrient_id] = np.nan

    specific = wide[NUTRIENT_IDS["energy_atwater_specific"]]
    general = wide[NUTRIENT_IDS["energy_atwater_general"]]
    legacy = wide[NUTRIENT_IDS["energy_legacy"]]

    wide["calories_per_100g"] = specific.combine_first(general).combine_first(legacy)

    wide["_energy_nutrient_id"] = np.select(
        [
            specific.notna(),
            general.notna(),
            legacy.notna(),
        ],
        [
            NUTRIENT_IDS["energy_atwater_specific"],
            NUTRIENT_IDS["energy_atwater_general"],
            NUTRIENT_IDS["energy_legacy"],
        ],
        default=np.nan,
    )

    wide["protein_grams_per_100g"] = wide[NUTRIENT_IDS["protein"]]
    wide["carbohydrate_grams_per_100g"] = wide[NUTRIENT_IDS["carbohydrate"]]
    wide["fat_grams_per_100g"] = wide[NUTRIENT_IDS["fat"]]

    return wide[[
        "fdc_id",
        "calories_per_100g",
        "protein_grams_per_100g",
        "carbohydrate_grams_per_100g",
        "fat_grams_per_100g",
        "_energy_nutrient_id",
    ]]


## 4. Source-specific metadata

We use a stable USDA identity where available:

- Foundation → NDB number when present, otherwise FDC ID.
- SR Legacy → NDB number when present, otherwise FDC ID.
- FNDDS → Food Code when present, otherwise FDC ID.

The raw FDC ID is still kept in the **stage** file for traceability.

Categories differ by USDA data type, so first we capture the original USDA category. A later function maps it into a small Spotter taxonomy.


In [ ]:
def load_food_master(directory: Path):
    path = find_csv(directory, "food.csv")
    columns = read_csv_columns(path)

    fdc_col = pick_column(columns, ["fdc_id"], context=str(path))
    description_col = pick_column(columns, ["description"], context=str(path))
    category_col = pick_column(
        columns,
        ["food_category_id"],
        required=False,
        context=str(path),
    )

    usecols = [fdc_col, description_col]
    if category_col:
        usecols.append(category_col)

    food = pd.read_csv(path, usecols=usecols, low_memory=False)

    rename = {
        fdc_col: "fdc_id",
        description_col: "description",
    }
    if category_col:
        rename[category_col] = "food_category_id"

    food = food.rename(columns=rename)
    food["fdc_id"] = pd.to_numeric(food["fdc_id"], errors="coerce").astype("Int64")

    return food


def load_sr_style_categories(directory: Path):
    path = find_csv(directory, "food_category.csv", required=False)
    if path is None:
        return None

    columns = read_csv_columns(path)
    id_col = pick_column(columns, ["id"], context=str(path))
    description_col = pick_column(columns, ["description"], context=str(path))

    categories = pd.read_csv(
        path,
        usecols=[id_col, description_col],
        low_memory=False,
    ).rename(columns={
        id_col: "food_category_id",
        description_col: "source_category",
    })

    categories["food_category_id"] = pd.to_numeric(
        categories["food_category_id"], errors="coerce"
    ).astype("Int64")

    return categories


def load_foundation_identity(directory: Path):
    path = find_csv(directory, "foundation_food.csv", required=False)
    if path is None:
        return None

    data = pd.read_csv(path, dtype="string", low_memory=False)
    columns = list(data.columns)

    fdc_col = pick_column(columns, ["fdc_id"], context=str(path))
    ndb_col = pick_column(
        columns,
        ["ndb_number", "NDB_number"],
        required=False,
        context=str(path),
    )

    result = pd.DataFrame({
        "fdc_id": pd.to_numeric(data[fdc_col], errors="coerce").astype("Int64"),
    })

    if ndb_col:
        result["_stable_source_id"] = data[ndb_col].astype("string").str.strip()
    else:
        result["_stable_source_id"] = result["fdc_id"].astype("string")

    return result


def load_sr_identity(directory: Path):
    path = find_csv(directory, "sr_legacy_food.csv", required=False)
    if path is None:
        return None

    data = pd.read_csv(path, dtype="string", low_memory=False)
    columns = list(data.columns)

    fdc_col = pick_column(columns, ["fdc_id"], context=str(path))
    ndb_col = pick_column(
        columns,
        ["ndb_number", "NDB_number"],
        required=False,
        context=str(path),
    )

    result = pd.DataFrame({
        "fdc_id": pd.to_numeric(data[fdc_col], errors="coerce").astype("Int64"),
    })

    if ndb_col:
        result["_stable_source_id"] = data[ndb_col].astype("string").str.strip()
    else:
        result["_stable_source_id"] = result["fdc_id"].astype("string")

    return result


def load_fndds_metadata(directory: Path):
    survey_path = find_csv(directory, "survey_fndds_food.csv")
    survey = pd.read_csv(survey_path, dtype="string", low_memory=False)
    columns = list(survey.columns)

    fdc_col = pick_column(columns, ["fdc_id"], context=str(survey_path))
    food_code_col = pick_column(
        columns,
        ["food_code"],
        required=False,
        context=str(survey_path),
    )
    category_number_col = pick_column(
        columns,
        ["wweia_category_number", "wweia_category_code"],
        required=False,
        context=str(survey_path),
    )

    result = pd.DataFrame({
        "fdc_id": pd.to_numeric(survey[fdc_col], errors="coerce").astype("Int64"),
    })

    if food_code_col:
        result["_stable_source_id"] = survey[food_code_col].astype("string").str.strip()
    else:
        result["_stable_source_id"] = result["fdc_id"].astype("string")

    if category_number_col:
        result["_category_join_key"] = normalize_join_key(survey[category_number_col])
    else:
        result["_category_join_key"] = pd.Series(pd.NA, index=result.index, dtype="string")

    category_path = find_csv(directory, "wweia_food_category.csv", required=False)
    if category_path is None:
        result["source_category"] = None
        return result

    categories = pd.read_csv(category_path, dtype="string", low_memory=False)
    category_columns = list(categories.columns)

    code_col = pick_column(
        category_columns,
        ["wweia_food_category_code", "wweia_category_number", "wweia_category_code"],
        context=str(category_path),
    )
    description_col = pick_column(
        category_columns,
        ["wweia_food_category_description", "description"],
        context=str(category_path),
    )

    category_lookup = pd.DataFrame({
        "_category_join_key": normalize_join_key(categories[code_col]),
        "source_category": categories[description_col].map(clean_text),
    }).drop_duplicates("_category_join_key")

    result = result.merge(
        category_lookup,
        on="_category_join_key",
        how="left",
        validate="many_to_one",
    )

    return result.drop(columns=["_category_join_key"])


## 5. Normalize USDA categories into a small Spotter category

This is intentionally **coarse**. The raw USDA category is preserved in `usda_foods_stage.csv`, so you can improve this mapping after EDA without losing information.

`MIXED_DISHES` is especially useful: because Spotter will have a separate `Recipe` entity, you may later decide not to import those rows into the `Food` catalog.


In [ ]:
CATEGORY_RULES = [
    ("MIXED_DISHES", [
        "mixed dish", "mixed dishes", "sandwich", "burger", "pizza",
        "soup", "stew", "casserole",
    ]),
    ("FATS_OILS", [
        "fats and oils", "fat and oil", "oils", "oil", "salad dressing",
    ]),
    ("SEAFOOD", [
        "finfish", "shellfish", "seafood", "fish",
    ]),
    ("POULTRY", [
        "poultry", "chicken", "turkey", "duck",
    ]),
    ("MEAT", [
        "beef", "pork", "lamb", "veal", "game meat", "sausages", "meat",
    ]),
    ("EGGS", [
        "egg",
    ]),
    ("DAIRY", [
        "dairy", "milk", "cheese", "yogurt", "yoghurt",
    ]),
    ("LEGUMES", [
        "legume", "beans", "peas", "lentil",
    ]),
    ("NUTS_SEEDS", [
        "nut and seed", "nuts", "seeds",
    ]),
    ("FRUIT", [
        "fruit",
    ]),
    ("VEGETABLE", [
        "vegetable", "potato",
    ]),
    ("GRAINS", [
        "cereal", "grain", "bread", "rice", "pasta", "noodle",
    ]),
    ("BEVERAGES", [
        "beverage", "coffee", "tea", "water", "juice",
    ]),
    ("SWEETS_SNACKS", [
        "sweet", "candy", "dessert", "cookie", "cake", "pastr",
        "snack", "sugar",
    ]),
    ("CONDIMENTS", [
        "spice", "herb", "sauce", "condiment",
    ]),
]


def normalize_category(source_category):
    value = clean_text(source_category)
    if not value:
        return None

    lowered = value.casefold()

    for spotter_category, keywords in CATEGORY_RULES:
        if any(keyword in lowered for keyword in keywords):
            return spotter_category

    return "OTHER"


## 6. Normalize one USDA source

Nothing is imputed.

Your Prisma model requires calories, protein, carbohydrate, and fat, so any row missing one of those values goes to the rejected file instead of silently becoming `0`.


In [ ]:
def normalize_source(source_name, config):
    directory = config["directory"]
    print(f"\n=== Normalizing {source_name} ===")

    food = load_food_master(directory)

    # Source-specific stable identity and category.
    if source_name == "FOUNDATION":
        identity = load_foundation_identity(directory)
        if identity is not None:
            food = food.merge(identity, on="fdc_id", how="left", validate="one_to_one")

        categories = load_sr_style_categories(directory)
        if categories is not None and "food_category_id" in food.columns:
            food = food.merge(
                categories,
                on="food_category_id",
                how="left",
                validate="many_to_one",
            )
        else:
            food["source_category"] = None

    elif source_name == "SR_LEGACY":
        identity = load_sr_identity(directory)
        if identity is not None:
            food = food.merge(identity, on="fdc_id", how="left", validate="one_to_one")

        categories = load_sr_style_categories(directory)
        if categories is not None and "food_category_id" in food.columns:
            food = food.merge(
                categories,
                on="food_category_id",
                how="left",
                validate="many_to_one",
            )
        else:
            food["source_category"] = None

    elif source_name == "FNDDS":
        metadata = load_fndds_metadata(directory)
        food = food.merge(metadata, on="fdc_id", how="left", validate="one_to_one")

    else:
        raise ValueError(f"Unsupported source: {source_name}")

    if "_stable_source_id" not in food.columns:
        food["_stable_source_id"] = food["fdc_id"].astype("string")

    # Fall back to FDC ID if the source-specific stable key is missing.
    food["_stable_source_id"] = (
        food["_stable_source_id"]
        .astype("string")
        .str.strip()
        .replace({"": pd.NA, "<NA>": pd.NA, "nan": pd.NA})
        .fillna(food["fdc_id"].astype("string"))
    )

    nutrient_values = load_target_nutrients(
        directory,
        wanted_fdc_ids=food["fdc_id"].dropna().tolist(),
    )

    stage = food.merge(
        nutrient_values,
        on="fdc_id",
        how="left",
        validate="one_to_one",
    )

    stage["name_en"] = stage["description"].map(clean_text)
    stage["name_ar"] = None

    # We do not invent aliases during ingestion.
    stage["aliases"] = None

    stage["category"] = stage["source_category"].map(normalize_category)

    stage["created_by_user_id"] = None
    stage["source"] = config["source"]
    stage["external_source_id"] = stage["_stable_source_id"].astype("string")
    stage["source_version"] = config["source_version"]
    stage["is_active"] = True
    stage["archived_at"] = None

    stage["_source_type"] = source_name
    stage["_name_key"] = stage["name_en"].map(normalized_name_key)

    numeric_columns = [
        "calories_per_100g",
        "protein_grams_per_100g",
        "carbohydrate_grams_per_100g",
        "fat_grams_per_100g",
    ]

    for column in numeric_columns:
        stage[column] = pd.to_numeric(stage[column], errors="coerce").round(2)

    print("Rows:", len(stage))
    print("Missing category:", stage["category"].isna().sum())
    print("Missing any required macro:", stage[numeric_columns].isna().any(axis=1).sum())

    return stage


## 7. Run all three sources


In [ ]:
normalized_sources = []

for source_name, config in SOURCES.items():
    normalized_sources.append(normalize_source(source_name, config))

stage = pd.concat(normalized_sources, ignore_index=True)

print("\nCombined rows:", len(stage))
display(stage[[
    "name_en",
    "source_category",
    "category",
    "calories_per_100g",
    "protein_grams_per_100g",
    "carbohydrate_grams_per_100g",
    "fat_grams_per_100g",
    "source",
    "external_source_id",
]].head(20))


## 8. Validation and rejected rows

A row is **not database-ready** when:

- name is missing
- name exceeds your Prisma `VarChar(200)`
- one of the four required nutrition values is missing
- nutrition contains a negative value
- source identity is missing

We do not silently repair these.


In [ ]:
REQUIRED_NUTRITION = [
    "calories_per_100g",
    "protein_grams_per_100g",
    "carbohydrate_grams_per_100g",
    "fat_grams_per_100g",
]


def rejection_reasons(row):
    reasons = []

    if not row["name_en"]:
        reasons.append("missing_name")
    elif len(row["name_en"]) > 200:
        reasons.append("name_over_200_chars")

    for column in REQUIRED_NUTRITION:
        value = row[column]
        if pd.isna(value):
            reasons.append(f"missing_{column}")
        elif value < 0:
            reasons.append(f"negative_{column}")

    if not row["external_source_id"] or pd.isna(row["external_source_id"]):
        reasons.append("missing_external_source_id")

    return "|".join(reasons)


stage["_rejection_reason"] = stage.apply(rejection_reasons, axis=1)

rejected = stage[stage["_rejection_reason"] != ""].copy()
accepted = stage[stage["_rejection_reason"] == ""].copy()

print("Accepted:", len(accepted))
print("Rejected:", len(rejected))

if not rejected.empty:
    display(
        rejected[[
            "name_en",
            "source",
            "external_source_id",
            "_rejection_reason",
        ]].head(30)
    )


## 9. QA checks

These are **warnings**, not automatic deletions.

For example, `900 kcal / 100g` can be plausible for nearly pure fat, so a high-calorie row should be inspected rather than automatically rejected.


In [ ]:
print("=== DATASET COUNTS ===")
display(
    accepted.groupby("source", dropna=False)
    .size()
    .rename("rows")
    .to_frame()
)

print("\n=== CATEGORY COUNTS ===")
display(
    accepted["category"]
    .value_counts(dropna=False)
    .rename_axis("category")
    .to_frame("rows")
)

print("\n=== ENERGY FIELD USED ===")
display(
    accepted["_energy_nutrient_id"]
    .value_counts(dropna=False)
    .rename_axis("nutrient_id")
    .to_frame("rows")
)

print("\n=== SUSPICIOUS CALORIES (> 1000 kcal / 100g) ===")
display(
    accepted.loc[
        accepted["calories_per_100g"] > 1000,
        ["name_en", "calories_per_100g", "source"]
    ].head(30)
)

print("\n=== MACROS > 100 g / 100g ===")
macro_outlier = (
    (accepted["protein_grams_per_100g"] > 100)
    | (accepted["carbohydrate_grams_per_100g"] > 100)
    | (accepted["fat_grams_per_100g"] > 100)
)

display(
    accepted.loc[
        macro_outlier,
        [
            "name_en",
            "protein_grams_per_100g",
            "carbohydrate_grams_per_100g",
            "fat_grams_per_100g",
            "source",
        ],
    ].head(30)
)


## 10. Duplicate candidates

We **flag**, rather than automatically remove, same-name foods from different USDA sources.

Why? `"Chicken breast, roasted"` appearing twice may genuinely refer to overlapping records, but preparation/state differences elsewhere are nutritionally meaningful and must not be collapsed with a careless `drop_duplicates()`.


In [ ]:
duplicate_mask = accepted.duplicated("_name_key", keep=False) & accepted["_name_key"].notna()

duplicate_candidates = (
    accepted.loc[
        duplicate_mask,
        [
            "_name_key",
            "name_en",
            "source",
            "external_source_id",
            "source_category",
            "category",
            "calories_per_100g",
            "protein_grams_per_100g",
            "carbohydrate_grams_per_100g",
            "fat_grams_per_100g",
        ],
    ]
    .sort_values(["_name_key", "source"])
    .reset_index(drop=True)
)

print("Rows participating in exact-name duplicate groups:", len(duplicate_candidates))
display(duplicate_candidates.head(50))


## 11. Optional exact-name deduplication

Leave this **off** for the first run.

After you inspect `usda_duplicate_name_candidates.csv`, you can turn it on.

Current priority:

1. Foundation — most current analytical source
2. FNDDS — useful consumed-food source
3. SR Legacy — broad historic fallback

Only identical normalized names are considered; names like `Potato, raw` and `Potato, boiled` remain different.


In [ ]:
if DEDUPLICATE_EXACT_NAMES:
    priority = {
        "USDA_FDC_FOUNDATION": 1,
        "USDA_FDC_FNDDS": 2,
        "USDA_FDC_SR_LEGACY": 3,
    }

    accepted["_source_priority"] = accepted["source"].map(priority).fillna(99)

    accepted = (
        accepted
        .sort_values(["_name_key", "_source_priority"])
        .drop_duplicates("_name_key", keep="first")
        .drop(columns=["_source_priority"])
        .reset_index(drop=True)
    )

    print("Rows after exact-name deduplication:", len(accepted))
else:
    print("Exact-name deduplication is OFF. No rows were removed.")


## 12. Build the exact database-import shape

Your database owns:

- `id`
- `created_at`
- `updated_at`

so the normalized source file does not create them.

`name_ar` and `aliases` are nullable for now.


In [ ]:
DB_COLUMNS = [
    "name_en",
    "name_ar",
    "aliases",
    "category",
    "calories_per_100g",
    "protein_grams_per_100g",
    "carbohydrate_grams_per_100g",
    "fat_grams_per_100g",
    "created_by_user_id",
    "source",
    "external_source_id",
    "source_version",
    "is_active",
    "archived_at",
]

foods_for_db = accepted[DB_COLUMNS].copy()

# Final uniqueness invariant expected by Prisma:
# @@unique([source, externalSourceId])
duplicate_source_ids = foods_for_db.duplicated(
    subset=["source", "external_source_id"],
    keep=False,
)

if duplicate_source_ids.any():
    bad = foods_for_db.loc[
        duplicate_source_ids,
        ["source", "external_source_id", "name_en"],
    ].sort_values(["source", "external_source_id"])

    display(bad.head(50))
    raise ValueError(
        "Duplicate (source, external_source_id) rows remain. "
        "Do not import until this is resolved."
    )

print("Final database-ready rows:", len(foods_for_db))
display(foods_for_db.head(20))


## 13. Save outputs


In [ ]:
STAGE_OUTPUT = OUTPUT_DIR / "usda_foods_stage.csv"
DB_OUTPUT = OUTPUT_DIR / "foods_for_db.csv"
REJECTED_OUTPUT = OUTPUT_DIR / "usda_foods_rejected.csv"
DUPLICATES_OUTPUT = OUTPUT_DIR / "usda_duplicate_name_candidates.csv"

# Stage includes diagnostic USDA columns useful during EDA.
stage.to_csv(STAGE_OUTPUT, index=False)

# Exact schema-oriented import file.
foods_for_db.to_csv(DB_OUTPUT, index=False)

rejected.to_csv(REJECTED_OUTPUT, index=False)
duplicate_candidates.to_csv(DUPLICATES_OUTPUT, index=False)

print("Saved:")
print(" ", STAGE_OUTPUT)
print(" ", DB_OUTPUT)
print(" ", REJECTED_OUTPUT)
print(" ", DUPLICATES_OUTPUT)


## 14. Final sanity summary


In [ ]:
summary = pd.DataFrame({
    "metric": [
        "raw normalized rows",
        "accepted rows",
        "rejected rows",
        "database-ready rows",
        "duplicate-candidate rows",
    ],
    "value": [
        len(stage),
        len(accepted),
        len(rejected),
        len(foods_for_db),
        len(duplicate_candidates),
    ],
})

display(summary)

print("\nRandom database-ready sample:")
display(
    foods_for_db.sample(
        min(20, len(foods_for_db)),
        random_state=42,
    )
)


# What to do after this notebook

Do **not** immediately seed `foods_for_db.csv`.

First inspect:

1. `usda_foods_rejected.csv`
2. `usda_duplicate_name_candidates.csv`
3. category counts
4. common USDA names
5. FNDDS `MIXED_DISHES` rows

Then decide what belongs in Spotter's initial global catalog.

Once that curation is settled, the next pipeline step is a small Node/Prisma importer that reads `foods_for_db.csv` and upserts on:

```text
(source, externalSourceId)
```

That makes future USDA refreshes idempotent rather than creating duplicate foods.
